In [ ]:
# ============================================================
# ROAD ACCIDENT SEVERITY PREDICTION USING MACHINE LEARNING
# ============================================================

# 1. IMPORT LIBRARIES
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


# ============================================================
# 2. CREATE SAMPLE ACCIDENT DATASET
# ============================================================

np.random.seed(42)

n = 1000

data = pd.DataFrame({
    "Weather": np.random.choice(
        ["Clear", "Rain", "Fog", "Storm"], n
    ),

    "Road_Condition": np.random.choice(
        ["Dry", "Wet", "Poor"], n
    ),

    "Road_Type": np.random.choice(
        ["Highway", "City Road", "Rural Road"], n
    ),

    "Vehicle_Type": np.random.choice(
        ["Car", "Bike", "Truck", "Bus"], n
    ),

    "Time_of_Day": np.random.choice(
        ["Day", "Night"], n
    ),

    "Number_of_Vehicles": np.random.randint(1, 6, n),

    "Speed": np.random.randint(20, 121, n),

    "Driver_Age": np.random.randint(18, 70, n)
})


# ============================================================
# 3. CREATE ACCIDENT SEVERITY (POINTS LOGIC)
# ============================================================

severity = []

for i in range(n):

    score = 0

    # Speed
    if data.loc[i, "Speed"] > 80:
        score += 2
    elif data.loc[i, "Speed"] > 60:
        score += 1

    # Road condition
    if data.loc[i, "Road_Condition"] == "Poor":
        score += 2
    elif data.loc[i, "Road_Condition"] == "Wet":
        score += 1

    # Weather
    if data.loc[i, "Weather"] in ["Fog", "Storm"]:
        score += 1

    # Time
    if data.loc[i, "Time_of_Day"] == "Night":
        score += 1

    # Number of vehicles
    if data.loc[i, "Number_of_Vehicles"] >= 4:
        score += 2
    elif data.loc[i, "Number_of_Vehicles"] >= 2:
        score += 1

    # Severity classification
    if score >= 5:
        severity.append("Fatal")
    elif score >= 3:
        severity.append("Serious")
    else:
        severity.append("Minor")


data["Accident_Severity"] = severity


# ============================================================
# 4. DISPLAY DATASET DETAILS
# ============================================================

print("FIRST 5 RECORDS:")
print(data.head())

print("\nDATASET SHAPE:")
print(data.shape)

print("\nMISSING VALUES:")
print(data.isnull().sum())

print("\nACCIDENT SEVERITY COUNT:")
print(data["Accident_Severity"].value_counts())


# ============================================================
# 5. CONVERT CATEGORICAL DATA INTO NUMBERS
# ============================================================

# Mapping categorical values manually
weather_map = {
    "Clear": 0,
    "Rain": 1,
    "Fog": 2,
    "Storm": 3
}

road_condition_map = {
    "Dry": 0,
    "Wet": 1,
    "Poor": 2
}

road_type_map = {
    "Highway": 0,
    "City Road": 1,
    "Rural Road": 2
}

vehicle_type_map = {
    "Car": 0,
    "Bike": 1,
    "Truck": 2,
    "Bus": 3
}

time_map = {
    "Day": 0,
    "Night": 1
}


data["Weather"] = data["Weather"].map(weather_map)
data["Road_Condition"] = data["Road_Condition"].map(road_condition_map)
data["Road_Type"] = data["Road_Type"].map(road_type_map)
data["Vehicle_Type"] = data["Vehicle_Type"].map(vehicle_type_map)
data["Time_of_Day"] = data["Time_of_Day"].map(time_map)

# Target encoding
severity_map = {
    "Minor": 0,
    "Serious": 1,
    "Fatal": 2
}

data["Accident_Severity"] = data["Accident_Severity"].map(severity_map)


# ============================================================
# 6. SEPARATE INPUT AND OUTPUT
# ============================================================

X = data.drop("Accident_Severity", axis=1)
y = data["Accident_Severity"]

print("\nINPUT FEATURES:")
print(X.columns)

print("\nTARGET:")
print("Accident_Severity")


# ============================================================
# 7. TRAIN-TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTRAINING DATA:", X_train.shape)
print("TESTING DATA:", X_test.shape)


# ============================================================
# 8. TRAIN RANDOM FOREST MODEL
# ============================================================

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

print("\nMODEL TRAINING COMPLETED!")


# ============================================================
# 9. PREDICTION ON TEST DATA
# ============================================================

y_pred = model.predict(X_test)


# ============================================================
# 10. MODEL ACCURACY
# ============================================================

accuracy = accuracy_score(y_test, y_pred)

print("\nMODEL ACCURACY:")
print(round(accuracy * 100, 2), "%")


# ============================================================
# 11. CLASSIFICATION REPORT
# ============================================================

print("\nCLASSIFICATION REPORT:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Minor", "Serious", "Fatal"]
    )
)


# ============================================================
# 12. CONFUSION MATRIX TEXT REPORT
# ============================================================

cm = confusion_matrix(y_test, y_pred)

print("\nCONFUSION MATRIX VALUES:")
print(cm)


# ============================================================
# 13. FEATURE IMPORTANCE ANALYSIS
# ============================================================

importance = pd.Series(
    model.feature_importances_,
    index=X.columns
)

importance = importance.sort_values(ascending=False)

print("\nFEATURE IMPORTANCE:")
print(importance)


# ============================================================
# 14. FEATURE IMPORTANCE BAR GRAPH
# ============================================================

importance.plot(
    kind="bar",
    figsize=(8, 5)
)

plt.title("Feature Importance")
plt.xlabel("Features")
plt.ylabel("Importance")
plt.show()


# ============================================================
# 15. USER INPUT FOR NEW ACCIDENT
# ============================================================

print("\n==========================================")
print("ENTER NEW ACCIDENT DETAILS")
print("==========================================")

weather = input("Weather (Clear/Rain/Fog/Storm): ")
road_condition = input("Road Condition (Dry/Wet/Poor): ")
road_type = input("Road Type (Highway/City Road/Rural Road): ")
vehicle_type = input("Vehicle Type (Car/Bike/Truck/Bus): ")
time_of_day = input("Time of Day (Day/Night): ")
number_of_vehicles = int(input("Number of Vehicles: "))
speed = int(input("Speed: "))
driver_age = int(input("Driver Age: "))


# ============================================================
# 16. CONVERT USER INPUT INTO NUMBERS & FIX PREDICTION ARRAY LOOKUP
# ============================================================

new_accident = pd.DataFrame([[
    weather_map[weather],
    road_condition_map[road_condition],
    road_type_map[road_type],
    vehicle_type_map[vehicle_type],
    time_map[time_of_day],
    number_of_vehicles,
    speed,
    driver_age
]], columns=X.columns)

# Run prediction for user input
user_pred = model.predict(new_accident)
reverse_severity_map = {0: "Minor", 1: "Serious", 2: "Fatal"}

# FIX: Added [0] to correctly extract the value from the NumPy array
predicted_label = reverse_severity_map[user_pred[0]].upper()

print("\n==========================================")
print("PREDICTED ACCIDENT SEVERITY:", predicted_label)
print("==========================================")